In [1]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
from diffusers import DDPMScheduler
import torch
from torch import nn

In [2]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text_encoder = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")
text_encoder.eval()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [3]:
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

In [9]:
def text_to_embedding(text):
  tokens = tokenizer(text, return_tensors="pt",padding=True, truncation=True)
  with torch.no_grad():
    outputs = text_encoder(**tokens,output_hidden_states=True)
  return outputs.hidden_states[-1]

In [5]:
def add_noise(embedding,timestep):
  noise = torch.rand_like(embedding)
  noisy_embedding = noise_scheduler.add_noise(embedding,noise,timestep)
  return noisy_embedding,noise


In [11]:
def show_noise_effect(embedding,timestep):
  noisy_embed,noise = add_noise(embedding,timestep)
  with torch.no_grad():
    logits = text_encoder(inputs_embeds=noisy_embed).logits
    noisy_text = tokenizer.batch_decode(logits.argmax(dim=-1)[0], skip_special_tokens=True)[0]
  return noisy_text


In [10]:
input_text = "The cat sat on the mat"
clean_embed = text_to_embedding(input_text)
clean_embed

tensor([[[-0.2027,  0.1071, -0.0085,  ..., -0.1198,  0.2286,  0.2522],
         [-0.3376, -0.2804, -0.1846,  ..., -0.3904,  0.9606, -0.6021],
         [-0.3106, -0.0439,  0.2795,  ..., -0.4990,  0.7590,  0.6673],
         ...,
         [-0.5056, -0.2047, -0.1147,  ..., -0.0776,  0.4049, -0.5230],
         [-0.6513, -0.4152, -0.3943,  ...,  0.6858,  0.1113, -0.0406],
         [ 0.7182,  0.1087, -0.3343,  ...,  0.0821, -0.4505, -0.5088]]])

In [12]:
for timestep in [200,500,800]:
  noisy_text = show_noise_effect(clean_embed,torch.tensor([timestep]))
  print(f"Timestep:{timestep}\nNoisy Text:{noisy_text}\n")

Timestep:200
Noisy Text:in in in in in in in in

Timestep:500
Noisy Text:and and and and and and and and

Timestep:800
Noisy Text:.. to as.. and them



In [13]:
denoiser = nn.Sequential(nn.Linear(768,768),nn.ReLU(),nn.Linear(768,768))

In [15]:
noisy_embed, true_noise = add_noise(clean_embed,torch.tensor(500))
print(noisy_embed)
print(true_noise)

tensor([[[ 0.2320,  0.5031,  0.0696,  ...,  0.3085,  0.6995,  0.5040],
         [ 0.5243, -0.0535, -0.0346,  ...,  0.2895,  1.0434,  0.3544],
         [ 0.4632,  0.1769,  0.6843,  ...,  0.8099,  1.1376,  0.8279],
         ...,
         [ 0.6898,  0.1506,  0.2721,  ...,  0.7947,  0.3989,  0.5331],
         [-0.0432,  0.4326, -0.0593,  ...,  0.2527,  0.9885,  0.3428],
         [ 0.4347,  0.3313,  0.7907,  ...,  0.9199,  0.5286, -0.0054]]])
tensor([[[0.3004, 0.4928, 0.0749,  ..., 0.3561, 0.6620, 0.4516],
         [0.6440, 0.0258, 0.0176,  ..., 0.4148, 0.8075, 0.5439],
         [0.5726, 0.1970, 0.6314,  ..., 0.9883, 0.9642, 0.6683],
         ...,
         [0.8652, 0.2163, 0.3166,  ..., 0.8501, 0.2978, 0.7071],
         [0.1442, 0.5711, 0.0528,  ..., 0.0639, 0.9970, 0.3688],
         [0.2441, 0.3134, 0.9204,  ..., 0.9340, 0.6813, 0.1422]]])


In [16]:
predicted_noise = denoiser(noisy_embed)
print(predicted_noise)

tensor([[[-0.0084, -0.0229,  0.1640,  ..., -0.0862,  0.0671, -0.2923],
         [ 0.0220,  0.1238,  0.1783,  ..., -0.1179,  0.1161, -0.3130],
         [ 0.0082,  0.0010, -0.0088,  ..., -0.0319,  0.1381, -0.3495],
         ...,
         [-0.0640,  0.0530,  0.1047,  ...,  0.0177,  0.1033, -0.1697],
         [ 0.0492, -0.0282,  0.1459,  ..., -0.1867,  0.1031, -0.2331],
         [ 0.0157,  0.0180,  0.0196,  ..., -0.0134,  0.2275, -0.2149]]],
       grad_fn=<ViewBackward0>)


In [17]:
predicted_clean = noisy_embed - predicted_noise
print(predicted_clean)

tensor([[[ 0.2404,  0.5260, -0.0945,  ...,  0.3948,  0.6324,  0.7963],
         [ 0.5023, -0.1773, -0.2128,  ...,  0.4074,  0.9273,  0.6674],
         [ 0.4550,  0.1759,  0.6931,  ...,  0.8418,  0.9995,  1.1774],
         ...,
         [ 0.7538,  0.0976,  0.1674,  ...,  0.7770,  0.2956,  0.7028],
         [-0.0923,  0.4608, -0.2052,  ...,  0.4394,  0.8854,  0.5759],
         [ 0.4190,  0.3133,  0.7710,  ...,  0.9332,  0.3011,  0.2095]]],
       grad_fn=<SubBackward0>)


In [19]:
with torch.no_grad():
  logits = text_encoder(inputs_embeds=predicted_clean).logits
  denoised_text = tokenizer.batch_decode(logits.argmax(dim=-1)[0], skip_special_tokens=True)[0]
print(denoised_text)

as on as as as in as on


In [20]:
show_noise_effect(clean_embed,torch.tensor(500))

'as old old old old is old old'